In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from deep_macrofin import PDEModel
from deep_macrofin import Comparator, OptimizerType
from tqdm import tqdm

In [ ]:
c_grid = [0.01, 1.99]
params = {
    "mu": 0.01,
    "siga": 0.25,
    "sigx": 0.12,
    "rho": -0.2,
    "carry_cost": 0.02,
    "alpha": 0.18,
    "phi": 0.002,
    "prop": 1.06,
    "liquidation": 100,
    "omega": 0.55,
    "r": 0.03,
    "I": 10,
    "xi": 0.015,
}

In [ ]:
def compute_hjb(c, F, F_Jac, F_Hess, r, mu, alpha, carry_cost, siga, sigx, rho):
    first_order = (alpha + c * (r - carry_cost - mu)) * F_Jac
    # need to change this for higher dimension
    second_order = (siga**2 * c**2 - 2 * rho * siga * sigx * c + sigx**2) * F_Hess.reshape(-1, 1) 
    value_term = (r - mu) * F
    return first_order + 0.5 * second_order - value_term    

def compute_bc(compute_F, c, p, phi, omega, alpha, r, mu):
    F = compute_F(c)
    first_term = torch.max(F - p * (c + phi), dim=0).values
    second_term = torch.tensor(omega * alpha / (r - mu), dtype=torch.float32, device=first_term.device)
    return torch.maximum(first_term, second_term).reshape(-1,1)

model = PDEModel("equity", config={"num_epochs": 10000, "optimizer_type": OptimizerType.Adam})
model.set_state(["c"], {"c": c_grid})
model.add_params(params)
model.register_functions([compute_hjb, compute_bc])
model.add_endog("F", config={"batch_jac_hes": True})
model.add_hjb_equation("compute_hjb(c, F, F_Jac, F_Hess, r, mu, alpha, carry_cost, siga, sigx, rho)")
model.add_endog_condition("F", 
                          "F(zero)", {"zero": torch.zeros(1, 1, dtype=torch.float32, device=model.device)},
                          Comparator.EQ,
                          "compute_bc(compute_F, c_lin, prop, phi, omega, alpha, r, mu)",
                          params | {"c_lin": torch.linspace(c_grid[0], 100, steps=10000, dtype=torch.float32, device=model.device).reshape(-1, 1)}
                        )
model.train_model("./models/decamps_equity", "model.pt", True)
model.load_model(torch.load("./models/decamps_equity/model_best.pt"))
model.eval_model(True)

In [ ]:
model.plot_vars(["F"], ncols=1)

In [ ]:
def compute_rhs(params, c_star):
    alpha = params["alpha"]
    r = params["r"]
    mu = params["mu"]
    lbd = params["carry_cost"]
    return 1 / (r-mu) * (alpha + c_star * (r - lbd - mu))

def root_finding(model: PDEModel, thres=1e-4):
    F = model.endog_vars["F"].model.eval()
    c = torch.tensor([0.5], requires_grad=True)  # example starting point
    optimizer = torch.optim.Adam([c], lr=0.01)
    loss = torch.tensor(torch.inf)
    for i in tqdm(range(100), desc=f"loss={loss.item()}, c={c.item()}"):
        optimizer.zero_grad()
        loss = (F(c) - compute_rhs(model.params, c)).pow(2).sum()  # squared error
        loss.backward()
        optimizer.step()
    return c.detach()
print(root_finding(model, 1e-3))